# L10c: Stochastic Multi-Armed Bandit Problems
In this lecture, we continue our exploration of Reinforcement Learning (RL), a branch of machine learning focused on how agents should take actions in an environment to maximize cumulative reward over time. We focus on multi-armed bandit problems, where the agent learns only from the reward of each action it takes.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Balance exploration and exploitation:__ An agent that learns from rewards must choose between trying actions to learn more about them (exploration) and taking the action that looks best so far (exploitation). We'll see why an agent needs both, and how each algorithm in this lecture balances them.
> * __Solve multi-armed bandit problems:__ In a multi-armed bandit problem, an agent repeatedly chooses one of several actions (arms), each with an unknown reward distribution, and tries to collect as much reward as possible. We'll work through the explore-first, epsilon-greedy, UCB1, and Thompson sampling algorithms for choosing the next arm.
> * __Compare bandit algorithms with regret:__ Regret is the reward an agent loses by not always choosing the best arm. We'll define the regret of a bandit algorithm and compare the expected-regret bounds for explore-first, epsilon-greedy, and UCB1.

At its core, RL is about learning from doing: an agent observes the state of its environment, takes actions, i.e., makes decisions, receives rewards, and updates its knowledge to improve future decision-making. Let’s get started!
___

## Exploration vs. Exploitation
In reinforcement learning, the problem on the surface is deceptively simple: an agent is in a state $s$ and can take an action $a\in A_{s}$, where $A_{s}$ is the set of actions currently available to the agent. The agent chooses an action $a$, implements it, receives a reward $r$, and transitions to a new state $s^{\prime}$. The goal is to learn a policy that maximizes cumulative reward over time, i.e., the best possible action in each state.

But when you think about it, the problem is actually quite complex. The agent must make decisions based on incomplete information and balance two competing objectives, exploration and exploitation:
1. **Exploration**: Trying new actions to discover potentially better rewards. This is essential for learning about the environment and finding optimal policies. Taking purely random actions, or actions that have not been tried often, to gather information about their outcomes and rewards is an example of exploration.
2. **Exploitation**: Leveraging current knowledge to choose the action that looks best right now. This involves choosing actions that have previously yielded high rewards based on the agent's experience. However, if the agent only exploits, it may miss out on discovering better actions that could yield higher rewards in the long run. The agent never tries anything new; how boring!

Striking the right balance between exploration and exploitation is crucial. If an agent explores too much, it may miss out on immediate rewards. If it exploits too much, it may fail to discover better long-term strategies; an action with a low observed reward may simply have been unlucky, especially if we have tried it only once. The algorithms in this lecture strike this balance in different ways: some explore on a fixed schedule, while others direct their exploration toward the actions whose rewards remain most uncertain.
___


## What Is a Bandit Problem?  
A bandit problem is a class of online (sequential) decision-making tasks in which an agent repeatedly chooses among $K$ options (called _arms_) and receives a reward based on that choice.

The agent chooses from $K$ alternatives (somehow) and executes the desired action. At each round $t$, the agent pulls one arm and observes a reward $r_{t}$. Good pulls yield higher rewards; poor pulls yield lower rewards (or even losses). The agent’s goal is to maximize cumulative reward over time.

Here are a few examples of applications of bandit problems:
* __Clinical Trials__: Balances learning about new treatments (exploration) with assigning patients to the current best therapy (exploitation).
* __Financial Portfolio Design__: Dynamically allocates capital across assets to maximize returns while testing novel investments.
* __Adaptive Routing__: Chooses network paths to minimize delay, trading off probing unknown routes against using established fast ones.
* __Recommendation Systems__: Iteratively selects items to display (like movies or products), balancing novel suggestions against proven favorites.

__Additional Resources__: Our lecture notes for this week were inspired by Chapter 1 of "Introduction to Multi-Armed Bandits" by Aleksandrs Slivkins. This is an excellent resource (albeit quite technical) for learning more about bandit problems. [The book is available online](https://arxiv.org/abs/1904.07272). We also drew material from the [Bandit problem Thompson sampling tutorial by Russo et al., 2020](https://arxiv.org/abs/1707.02038).

In the **stationary stochastic bandit problem**, the agent has a fixed set of arms $\mathcal{A}$, with $|\mathcal{A}|=K$, and plays for $T$ rounds. Each arm $a$ has an unknown reward distribution $\mathcal{D}_a$ supported on $[0,1]$. Every pull independently draws a reward from that arm's fixed distribution. The regret bounds below rely on these three assumptions: independent draws, distributions that do not change (stationary), and rewards between 0 and 1.

For $t=1,2,\dots,T$ __do__:
1. The agent chooses an arm $a_t\in\mathcal{A}$ using its previous actions and observed rewards.
2. The environment independently draws $r_t\sim\mathcal{D}_{a_t}$.
3. The agent receives $r_t$ and updates its estimates using that observation.

The agent observes only the reward of the selected arm; it never sees the distribution $\mathcal{D}_a$. The unknown mean reward of arm $a$ is $\mu(a)=\mathbb{E}_{r\sim\mathcal{D}_a}[r]$. The goal of the agent is to maximize its total reward over the $T$ rounds. However, the goal of the algorithm designer is to minimize the _regret_ of the algorithm that the agent uses to choose its arms.


### Regret
Regret measures the difference between what could have been achieved by always making the best decision and what the agent actually received. Let $\mu^\star=\max_{a\in\mathcal{A}}\mu(a)$ be the highest **true mean reward**. Always pulling the best arm earns $T\mu^\star$ in expectation over $T$ rounds, and the regret of an observed game is given by:
$$
R(T)=T\mu^\star-\sum_{t=1}^{T}r_t.
$$
The first term is an expected reward, while the second term adds up the rewards actually received. So random reward fluctuations move $R(T)$ up and down, even after the agent has learned to pull the best arm.

When we know the arm means, we can also evaluate the **cumulative gap loss**, often called pseudo-regret. Define the gap of arm $a$ as $\Delta(a)=\mu^\star-\mu(a)$; then the cumulative gap loss is given by:
$$
G(T)=\sum_{t=1}^{T}\Delta(a_t)=\sum_{t=1}^{T}\left(\mu^\star-\mu(a_t)\right).
$$
Each pull adds the mean reward given up by choosing that arm, so a pull of the best arm adds nothing. The gap loss still varies from game to game, because the arms an algorithm selects depend on the rewards it has observed.

Once arm $a_t$ is selected, its expected reward is $\mu(a_t)$. Taking expectations gives:
$$
\mathbb{E}[R(T)]=\mathbb{E}[G(T)].
$$
The theoretical bounds below concern this expected regret, averaged over the reward draws and the algorithm's own random choices. In the L10d lab, we look at one game first, then average many games to estimate the expected regret.


### Explore-First Exploration
A straightforward approach to the multi-armed bandit problem is to explore each arm equally. This is called _uniform exploration_ or the **explore-first algorithm**. The agent begins with a purely _exploration phase_, pulling each arm $N$ times. After this exploration phase, the agent pulls the arm with the highest sample mean reward for the rest of the game. This is called the _exploitation phase_.

#### Explore-First Algorithm
__Initialize__: Supply $K$ arms and a horizon $T\geq K$. Choose the number of pulls per arm, an integer $N$ with $1\leq N\leq\lfloor T/K\rfloor$. We use:
$$
N=\min\left\{\left\lfloor\frac{T}{K}\right\rfloor,\;\max\left\{1,\;\operatorname{round}\left[\left(\frac{T}{K}\right)^{2/3}(\log T)^{1/3}\right]\right\}\right\}.
$$
The $\max$ makes sure every arm is pulled at least once, and the $\min$ makes sure the exploration fits in $T$ rounds.

1. __Explore__: Pull each arm $N$ times. Divide the sum of each arm's rewards by $N$ to obtain its sample mean.
2. __Select__: Choose the arm with the highest sample mean, breaking ties by the first arm index.
3. __Exploit__: Pull that arm for the remaining $T-KN$ rounds.

__Expected-regret bound__: Under the stated stochastic reward assumptions, an exploration count of order $(T/K)^{2/3}(\log T)^{1/3}$ gives expected regret of order $T^{2/3}(K\log T)^{1/3}$. [Slivkins, Section 1.2](https://arxiv.org/abs/1904.07272) derives this result by balancing the cost of exploration against the cost of committing to a poorly estimated arm.


### Epsilon-Greedy Exploration
One issue with uniform exploration is that the exploration phase may be costly if many of the arms have a large gap $\Delta(a)$, because every exploration pull of such an arm gives up $\Delta(a)$ in expected reward. In that case, it may be better to spread out (and interleave) exploration and exploitation. This is the idea behind the **epsilon-greedy algorithm**: the agent pulls the best arm so far with probability $1-\epsilon_t$ and a random arm with probability $\epsilon_t$, where the exploration probability $\epsilon_t$ shrinks as observations accumulate. In other sources, $\epsilon$ is a constant, not a function of $t$.

#### Epsilon-Greedy Algorithm
__Initialize__: Supply $K$ arms and a horizon $T\geq K$. Pull each arm once and record its reward sum, pull count, and sample mean.

For $t=K+1,K+2,\dots,T$ __do__:
1. Compute the exploration probability and draw $u$ uniformly from $[0,1]$. Our exploration probability is given by:
   $$
   \epsilon_t=\min\left\{1,\left(\frac{K\log t}{t}\right)^{1/3}\right\}.
   $$
2. __If__ $u\leq\epsilon_t$, __then__ choose an arm uniformly at random.
3. __Else__, choose the arm with the highest sample mean, breaking ties by the first arm index.
4. Pull the selected arm, observe its reward, and update its reward sum and pull count.

The cap keeps $\epsilon_t$ a valid probability in the early rounds, when $K\log t>t$. A random pull can also land on the current best arm.

__Theorem__: The epsilon-greedy algorithm with exploration probability $\epsilon_t=t^{-1/3}(K\log t)^{1/3}$ achieves the expected-regret bound $\mathbb{E}[R(t)]\leq t^{2/3}\cdot\mathcal{O}(K\log t)^{1/3}$ for each round $t$ ([Slivkins, Theorem 1.6](https://arxiv.org/abs/1904.07272)). Pulling each arm once first and capping $\epsilon_t$ at one do not change this order. The L10d lab uses this same schedule.


### Optimism Under Uncertainty
Let's consider an approach that adapts exploration to the observed rewards: the optimism under uncertainty algorithm. The key assumption of this approach:
* __Assumption__: Assume each arm is as good as it can be given the observations so far, and choose the best arm based on these optimistic estimates. This intuition leads to the `UCB1` algorithm.

Given a history of rewards and the number of pulls for each arm, the `UCB1` algorithm calculates an upper confidence bound (UCB) and uses it to decide which arm to pull. 

__Definition__: Upper Confidence Bound (UCB). During each round $t =1,2, \dots, T$ the `UCB1` algorithm maximizes the sum $\bar{\mu}(a)+U(a,t)$ where $\bar{\mu}(a)$ is the _estimated_ mean reward of arm $a\in\mathcal{A}$ at time $t$ and $U(a,t)$ is the _confidence radius_ of arm $a\in\mathcal{A}$ at time $t$, so their sum is an upper confidence bound on the mean reward. The confidence radius is given by:
$$
\begin{align*}
U(a,t) = \sqrt{\frac{2\log(t)}{N_{a}}},
\end{align*}
$$
where $N_{a}$ is the number of times that arm $a\in\mathcal{A}$ has been pulled before round $t$. The `UCB1` algorithm was originally proposed by Auer, Cesa-Bianchi, and Fischer in 2002.

#### UCB1 Algorithm
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$.

__Initialize__: Pull each arm $a\in\mathcal{A}$ once and record the rewards. For each arm $a\in\mathcal{A}$, set $N_{a} = 1$ and $\bar{\mu}(a) = r_{a}$.

For rounds $t = K+1,K+2,\dots,T$ __do__:
1. Compute the confidence radius $U(a,t)$ for each arm $a\in\mathcal{A}$.
2. Choose the best arm $a^{\star} = \text{arg}\max\,\left\{\bar{\mu}(a)+U(a,t)\mid\,a\in\mathcal{A}\right\}$ at time $t$.
3. Execute action $a^{\star}$ and receive a reward $r_{t}$ from the _adversary_ (nature).
4. Increment the pull count $N_{a^{\star}}$ and update the estimated mean reward $\bar{\mu}(a^{\star})$, the sum of the arm's rewards divided by its pull count.

__Theorem__: Under the stated stochastic reward assumptions, the `UCB1` algorithm achieves a regret bound for $K$ arms of $\mathbb{E}\left[R(T)\right]\leq\mathcal{O}\left(\sqrt{KT\cdot\log(T)}\right)$ 
over $T$ rounds.

#### Worked Example: An Uncertain Arm Receives Another Pull
Suppose two arms have been pulled for five rounds. Arm 1 has been pulled four times and returned rewards summing to 3; arm 2 has been pulled once and returned zero. Before round $t=6$, the sample means and upper confidence bounds are:

| Arm | Pull count | Reward sum | Sample mean | Confidence radius | Upper confidence bound |
|---|---:|---:|---:|---:|---:|
| 1 | 4 | 3 | 0.750 | 0.947 | 1.697 |
| 2 | 1 | 0 | 0.000 | 1.893 | 1.893 |

For example, arm 2's upper confidence bound is given by:
$$
0+\sqrt{\frac{2\log 6}{1}}\approx1.893.
$$
The algorithm selects arm 2, even though its sample mean is lower, because one observation leaves more uncertainty than four. If its next reward is 1, its pull count becomes 2, its reward sum becomes 1, and its sample mean becomes $1/2$. The confidence bonus is an allowance for uncertainty, so the upper confidence bound can exceed the maximum possible single reward of one.

___

## Binary Bernoulli Bandit Problem
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary, $r_{t} \in \left\{0,1\right\}$. The probability of getting reward `1` is unknown and needs to be estimated. The goal is to maximize the expected reward by selecting the best action at each time step.

* _Difference_: Unlike a completely general stochastic bandit problem, in the binary Bernoulli bandit problem the agent can _model how the world responds_ using a (deceptively) simple reward distribution, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). Thus, _the agent has a model of the world_ (which is so super cool!).
* _Binary_: The reward distribution is binary, but the experiment represented by the action $a$ can be a complex statement or function that _evaluates_ to a boolean value. Thus, we can model many complex scenarios that evaluate to `true` or `false`.

The Bernoulli distribution is a discrete probability distribution that returns a value of `1` with probability $\theta$ and value `0` with probability $1-\theta$. The probability mass function of the Bernoulli distribution is given by:
$$
\begin{equation*}
\texttt{Bern}(r; \theta) = \begin{cases}
\theta & \text{if } r = 1\\
1-\theta & \text{if } r = 0
\end{cases},
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward and $\theta\in[0,1]$ is the probability of getting reward `r = 1`. If $r\sim\texttt{Bern}(\theta)$, the expected reward is $\mathbb{E}[r] = \theta$ and its variance is $\text{Var}[r] = \theta(1-\theta)$.
* _Ready to get your mind blown_? Ok, so here is the _cool part_: the agent models the parameter $\theta$ using a _probability distribution_ (e.g., [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution)) and updates this distribution as it observes rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874).

The [Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) is a natural choice for this model:

> __Beta distribution:__ The distribution $\text{Beta}(\alpha,\beta)$ describes an unknown probability $\theta\in[0,1]$. The parameters $\alpha>0$ and $\beta>0$ set its shape, and its mean is $\alpha/(\alpha+\beta)$. Suppose our belief about $\theta$ before we see any rewards (the __prior__) is $\text{Beta}(\alpha,\beta)$, where $\text{Beta}(1,1)$ is the uniform distribution on $[0,1]$. After we observe $S$ successes and $F$ failures, our updated belief (the __posterior__) is again a Beta distribution, $\text{Beta}(\alpha+S,\,\beta+F)$.

Yeah. That's cool. But how do we solve this problem?

### $\epsilon$-Greedy Binary Bernoulli Bandit
The $\epsilon$-greedy algorithm is simple and effective for solving the binary Bernoulli bandit problem.

The algorithm exploits the arm with the highest posterior mean with probability $1-\epsilon_t$ and explores by choosing an arm uniformly at random with probability $\epsilon_t$. The pseudocode for the $\epsilon$-greedy algorithm is given below [(a more detailed version can be found here)](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-7/L7c/docs/BBBPcode.pdf):

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$.

__Initialize__: Set the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha(a) = 1$ and $\beta(a) = 1$, and set the number of successes and failures to $S(a) = F(a) = 0$. Pull each arm once and update its success and failure counts.

For $t = K+1,K+2,\dots,T$ __do__:
1. Draw a random number $u\in\left[0,1\right]$ and compute a threshold $\epsilon_t=\min\left\{1,\left(K\log(t)/t\right)^{1/3}\right\}$.
2. __Exploration__: __If__ $u\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute the action $a_{t}$ and receive a reward $r_{t} \in \left\{0,1\right\}$ from the _adversary_ (nature).
3. __Exploitation__: __Else if__ $u>\epsilon_{t}$, choose action $a_{t} = a^{\star}$, the action with the _highest expected probability of success_ (greedy choice), using the agent's model of the world. The highest probability action is: $a^{\star} = \arg\max_{a\in\mathcal{A}}\left\{\frac{\alpha(a) + S(a)}{\alpha(a) + \beta(a) + S(a) + F(a)}\right\}$, the posterior mean, where $S(a)$ and $F(a)$ are the number of successes and failures for arm $a$. Execute the action $a_{t}$ and receive a reward $r_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the number of successes $S(a_{t})$ and failures $F(a_{t})$ for the arm $a_{t}$ chosen in step 2 or 3, using its reward $r_{t}$:
$$
\begin{equation*}
S(a_{t}) \gets S(a_{t}) + r_{t},\quad F(a_{t}) \gets F(a_{t}) + (1-r_{t})
.
\end{equation*}
$$

Using a model of the world allows the agent to make decisions about which actions to take. The agent keeps a Beta distribution for the success probability of _each_ action, and when it exploits, it pulls the arm with the highest posterior mean.

### Thompson Sampling Binary Bernoulli Bandit
An alternative Bayesian approach is Thompson sampling, which samples from the posterior distribution to make decisions.

> The algorithm selects the _best action_ by sampling from the posterior distribution for each arm and choosing the one with the highest sample.

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$.

__Initialize__: Set the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha(a) = 1$ and $\beta(a) = 1$, and set the number of successes and failures to $S(a) = F(a) = 0$.

For $t = 1,2,\dots,T$ __do__:
1. Sample from the posterior for each arm: $p(a)\sim\text{Beta}\left(\alpha(a)+S(a),\,\beta(a)+F(a)\right)$ for every $a\in\mathcal{A}$, where $S(a)$ and $F(a)$ are the number of successes and failures for arm $a$.
2. Choose the action with the highest sampled probability: $a_{t} = \arg\max_{a\in\mathcal{A}}\left\{p(a)\right\}$.
3. Execute the action $a_{t}$ and receive a reward $r_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the number of successes $S(a_{t})$ and failures $F(a_{t})$ for the chosen arm $a_{t}$ using the reward $r_{t}$:
$$
\begin{equation*}
S(a_{t}) \gets S(a_{t}) + r_{t},\quad F(a_{t}) \gets F(a_{t}) + (1-r_{t})
.
\end{equation*}
$$

Thompson sampling is fully Bayesian and often performs better than $\epsilon$-greedy in practice, because its exploration goes mostly to arms that could still be the best, rather than uniformly to every arm ([Russo et al., 2020](https://arxiv.org/abs/1707.02038)).

#### Worked Example: Updating a Bernoulli Arm
With uniform $\operatorname{Beta}(1,1)$ priors, suppose arm 1 has three successes and one failure, while arm 2 has no successes and one failure. Their posteriors are $\operatorname{Beta}(4,2)$ and $\operatorname{Beta}(1,2)$, with means $2/3$ and $1/3$.

Suppose the next posterior samples are 0.55 for arm 1 and 0.70 for arm 2. Thompson sampling chooses arm 2 because its sample is larger. If that pull succeeds, arm 2 now has one success and one failure, and its posterior becomes $\operatorname{Beta}(2,2)$ with mean $1/2$. Arm 1's posterior stays unchanged because we received no new observation from that arm. Sampling lets an uncertain arm receive another trial even when its posterior mean is lower.

___

## Lab
In lab `L10d`, we run the explore-first, epsilon-greedy, and UCB1 algorithms on a bandit with seven arms and compare their regret with the bounds above. We then implement Thompson sampling for a binary Bernoulli bandit and watch the posterior of the best arm concentrate around its true success probability, while the arms it rarely pulls stay uncertain.

___

## Summary
In this lecture, we introduced the stochastic multi-armed bandit problem and four algorithms for balancing exploration and exploitation.

> __Key Takeaways:__
>
> * __Exploration and exploitation:__ We saw that an agent learning from rewards must both try actions to learn about them and take the action that looks best so far. Too much exploration wastes rewards on poor arms, while too little can lock the agent onto an arm that only looked good by chance.
>
> * __Bandit algorithms:__ We worked through four ways to choose the next arm: explore every arm first and then commit, explore at random with a shrinking probability (epsilon-greedy), add an optimism bonus to each arm's estimated reward (UCB1), and sample from a Beta model of each arm's success probability (Thompson sampling). For binary rewards, the Beta model turns the successes and failures of each arm into a distribution for its success probability, which the agent updates after every pull.
>
> * __Regret bounds:__ We measured each algorithm by its regret, the reward lost by not always choosing the best arm. The explore-first and epsilon-greedy bounds grow with the two-thirds power of the number of rounds, while the UCB1 bound grows only with about the square root, so UCB1 has the strongest guarantee of the three.

Next week, the agent's actions also change the state of its environment, which leads to Markov decision processes.

___